# M15b · Arch 4B: frozen CLIP + private bottleneck

See `docs4/arch4b_frozen_bottleneck.tex`. No LN tuning. Private U_g in bottleneck; discarded at deploy.

**KEEP** iff NT ≥ 81.21 and DFD ≥ 95.5.


## 1. Setup
### 1.1 Settings


In [12]:
# 1.1 Settings
import json, os
from pathlib import Path

DATA_ROOT = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
FACES_ROOT = DATA_ROOT / "faces"
INDEX_CSV = DATA_ROOT / "index" / "ffpp_c23.csv"
FACES_DFD = DATA_ROOT / "faces_dfd"
DFD_INDEX = DATA_ROOT / "index" / "dfd_c23.csv"
FACES_UADFV = DATA_ROOT / "faces_uadfv"
UADFV_INDEX = DATA_ROOT / "index" / "uadfv.csv"
RUNS_ROOT = DATA_ROOT / "runs"
M15B_DIR = RUNS_ROOT / "m15b"
M11A_DIR = RUNS_ROOT / "m11a"
M4_DIR = RUNS_ROOT / "m4"

ALL_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
TRAIN_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
SHORT = {"original": "real", "Deepfakes": "DF", "Face2Face": "F2F", "FaceSwap": "FS",
         "NeuralTextures": "NT", "FaceShifter": "FSh"}
G = len(TRAIN_GENERATORS)

ARCH, PRETRAINED = "ViT-L-14", "openai"
W_UNIFORM, W_ALIGN = 0.5, 0.1
AUGMENT = True
BATCH_SIZE = 128
EPOCHS = 10
CYCLE_EPOCHS = 10
WARMUP_EPOCHS = 1
LR_MAX, LR_MIN = 3e-4, 1e-5
SEED = 0
BOT_DIM = 64
PRIV_RANK = 16
LAMBDA = 0.01
RUN_DFD, RUN_UADFV = True, True

A0 = {"seen": 94.54, "DF": 99.49, "F2F": 97.86, "FS": 99.60, "NT": 81.21, "FSh": 92.25, "DFD": 95.15}
REFS = {
    "full LayerNorms (M4 λ=∞)": (M4_DIR / f"gbla_laminf_{ARCH}_seed{SEED}", 96.05, 99.38),
    "GBLA λ=0.01 (M4) / Arch1": (M4_DIR / f"gbla_lam0.01_{ARCH}_seed{SEED}", 96.71, 99.63),
}
VAL_FRAMES_PER_VIDEO = 8
CACHE_IN_RAM = True
EVAL_BATCH = 256

COMMON_CONFIG = {
    "architecture": "Arch4B", "notebook": "M15b",
    "bot_dim": BOT_DIM, "priv_rank": PRIV_RANK, "lambda": LAMBDA,
    "private": "bottleneck adapters U_g (train only)",
    "docs": "docs4/arch4b_frozen_bottleneck.tex",
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "seed": SEED,
    "lr_max": LR_MAX, "lr_min": LR_MIN, "augment": AUGMENT, "arch": ARCH,
}
assert INDEX_CSV.exists()
M15B_DIR.mkdir(parents=True, exist_ok=True)
print("M15b ->", M15B_DIR, "bot_dim", BOT_DIM, "priv_rank", PRIV_RANK)



M15b -> /nlsasfs/home/isea/isea64/GBLA-Deepfake Detection/runs/m15b bot_dim 64 priv_rank 16


### 1.2 Imports


In [13]:
# 1.2 Imports and environment
import json, math, random, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms import v2 as T
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
assert torch.cuda.is_available(), "M15b needs the GPU"
DEVICE = "cuda"


def usable_cpus():
    """CPUs this job may use: the affinity mask and the cgroup CPU quota."""
    n = len(os.sched_getaffinity(0))
    for quota_file, period_file in [("/sys/fs/cgroup/cpu.max", None),
                                    ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")]:
        try:
            if period_file is None:
                quota, period = Path(quota_file).read_text().split()
            else:
                quota, period = Path(quota_file).read_text().strip(), Path(period_file).read_text().strip()
            if quota not in ("max", "-1"):
                n = min(n, max(1, int(quota) // int(period)))
        except (OSError, ValueError):
            pass
    return n


def available_memory_gb():
    """Free RAM for this job: the smaller of MemAvailable and the cgroup limit minus non-cache usage."""
    candidates = []
    try:
        meminfo = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
        candidates.append(int(meminfo["MemAvailable"].split()[0]) * 1024)
    except (OSError, KeyError, ValueError):
        pass
    for limit_file, usage_file, stat_file, cache_key in [
            ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.stat", "file"),
            ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes",
             "/sys/fs/cgroup/memory/memory.stat", "cache")]:
        try:
            limit = Path(limit_file).read_text().strip()
            if not limit.isdigit() or int(limit) >= 1 << 60:
                continue
            usage = int(Path(usage_file).read_text().strip())
            stat = dict(line.split() for line in Path(stat_file).read_text().splitlines())
            candidates.append(int(limit) - usage + int(stat.get(cache_key, 0)))
        except (OSError, ValueError):
            pass
    return min(candidates) / 1e9 if candidates else 0.0


CPUS = usable_cpus()
NUM_WORKERS = max(1, min(16, CPUS - 1))     # one CPU stays with the training process
torch.set_num_threads(2)

pd.Series({
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "open_clip": open_clip.__version__,
    "GPU": torch.cuda.get_device_name(0),
    "CPU cores on the node": os.cpu_count(),
    "CPUs usable by this job": CPUS,
    "data loader workers": NUM_WORKERS,
    "free RAM for this job (GB)": round(available_memory_gb(), 1),
}, name="value").to_frame()


,value
torch,2.6.0+cu124
torchvision,0.21.0+cu124
open_clip,3.3.0
GPU,NVIDIA A100-SXM4-40GB
CPU cores on the node,256
CPUs usable by this job,1
data loader workers,1
free RAM for this job (GB),925.7


## 2. Data
### 2.1


In [14]:
# 2.1 Index and subsets
index = pd.read_csv(INDEX_CSV, dtype={"video_id": str, "target_id": str, "source_id": str},
                    keep_default_na=False)
CATEGORIES = ["original"] + ALL_GENERATORS
GEN_CODE = {g: k for k, g in enumerate(CATEGORIES)}


def subset(split, generators):
    keep = (index.split == split) & index.generator.isin(["original", *generators])
    return index[keep].reset_index(drop=True)


train_df = subset("train", TRAIN_GENERATORS)
val_all = subset("val", TRAIN_GENERATORS)
frames_per_video = int(val_all.groupby(["generator", "video_id"]).size().max())
keep_every = max(1, frames_per_video // VAL_FRAMES_PER_VIDEO)
frame_rank = val_all.groupby(["generator", "video_id"]).frame_idx.rank(method="first").astype(int) - 1
val_df = val_all[frame_rank % keep_every == 0].reset_index(drop=True)
test_df = subset("test", ALL_GENERATORS)

display(pd.DataFrame({name: df.groupby("generator").size().reindex(CATEGORIES)
                      for name, df in [("train", train_df), ("val (subsampled)", val_df), ("test", test_df)]}
                     ).fillna(0).astype(int))


,train,val (subsampled),test
generator,,,
original,23040,1120,4480
Deepfakes,23040,1120,4480
Face2Face,23040,1120,4480
FaceSwap,23040,1120,4480
NeuralTextures,23040,1120,4480
FaceShifter,0,0,4480


### 2.2


In [15]:
# 2.2 Decoded crops in RAM
def load_images(df, desc):
    """Decodes every crop of an index slice into one uint8 array (N, H, W, 3), using threads."""
    paths = [FACES_ROOT / p for p in df.path]
    first = np.asarray(Image.open(paths[0]).convert("RGB"))
    out = np.empty((len(paths), *first.shape), dtype=np.uint8)

    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))

    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out


images = {}
budget = 0.6 * available_memory_gb()
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    need = len(df) * 224 * 224 * 3 / 1e9
    if CACHE_IN_RAM and need <= budget:
        t = time.time()
        images[name] = load_images(df, f"{name} crops -> RAM")
        budget -= need
        sec = time.time() - t
        print(f"{name}: {len(df):,} crops, {need:.1f} GB in RAM, {sec / 60:.1f} min ({len(df) / sec:.0f} crops/s from disk)")
    else:
        images[name] = None
        print(f"{name}: read from disk during training (needs {need:.1f} GB, RAM budget left {budget:.1f} GB)")


train crops -> RAM: 100%|██████████| 115200/115200 [08:26<00:00, 227.44it/s]


train: 115,200 crops, 17.3 GB in RAM, 8.5 min (226 crops/s from disk)


val crops -> RAM: 100%|██████████| 5600/5600 [00:29<00:00, 191.48it/s]


val: 5,600 crops, 0.8 GB in RAM, 0.5 min (190 crops/s from disk)


test crops -> RAM: 100%|██████████| 26880/26880 [02:18<00:00, 194.41it/s]

test: 26,880 crops, 4.0 GB in RAM, 2.3 min (194 crops/s from disk)


### 2.3


In [16]:
# 2.3 Round-robin loader
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def make_transform(train):
    steps = []
    if train and AUGMENT:
        steps += [
            T.RandomHorizontalFlip(0.5),
            T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.1),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.JPEG(quality=(40, 100)),
        ]
    steps += [T.ToDtype(torch.float32, scale=True), T.Normalize(CLIP_MEAN, CLIP_STD)]
    return T.Compose(steps)


class FaceDataset(Dataset):
    def __init__(self, df, train, images=None):
        self.paths = [str(FACES_ROOT / p) for p in df.path]
        self.images = images
        self.labels = torch.tensor(df.label.values, dtype=torch.long)
        self.gens = torch.tensor(df.generator.map(GEN_CODE).values, dtype=torch.long)
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.images is not None:
            image = torch.from_numpy(np.array(self.images[i])).permute(2, 0, 1)
        else:
            image = read_image(self.paths[i], mode=ImageReadMode.RGB)
        return self.transform(tv_tensors.Image(image)).as_subclass(torch.Tensor), self.labels[i], self.gens[i]


class RoundRobinSampler(Sampler):
    """One epoch = one block per training generator, in an order fixed by (seed, epoch)."""

    def __init__(self, df, seed):
        self.seed = seed
        self.epoch = 0
        gen = df.generator.values
        real_idx = np.flatnonzero(gen == "original")
        # fixed split: real row r goes to generator r % G
        self.blocks = {}
        for k, g in enumerate(TRAIN_GENERATORS):
            fakes = np.flatnonzero(gen == g)
            reals = real_idx[k::G][: len(fakes) // G]
            self.blocks[g] = (fakes, reals)
        self.n_per = len(self.blocks[TRAIN_GENERATORS[0]][0]) + len(self.blocks[TRAIN_GENERATORS[0]][1])

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        order_rng = np.random.default_rng([self.seed, self.epoch, 1])
        gens = list(TRAIN_GENERATORS)
        order_rng.shuffle(gens)
        out = []
        for g in gens:
            fakes, reals = self.blocks[g]
            rng = np.random.default_rng([self.seed, ALL_GENERATORS.index(g), self.epoch])
            out.extend(rng.permutation(np.concatenate([fakes, reals])).tolist())
        return iter(out)

    def __len__(self):
        return self.n_per * G


train_ds = FaceDataset(train_df, train=True, images=images["train"])
train_sampler = RoundRobinSampler(train_df, SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
owned = {SHORT[g]: (len(f), len(r)) for g, (f, r) in train_sampler.blocks.items()}
print("fakes / reals owned by each generator:", owned)
print(f"steps per epoch: {len(train_loader)}  (M1 was 900);  workers: {NUM_WORKERS}")
assert len(train_loader) == 900, len(train_loader)


fakes / reals owned by each generator: {'DF': (23040, 5760), 'F2F': (23040, 5760), 'FS': (23040, 5760), 'NT': (23040, 5760)}
steps per epoch: 900  (M1 was 900);  workers: 1


### 2.4


In [17]:
# 2.4 Data-loading speed
def loader_rate(loader, n_batches=30):
    """Steady-state images/s after the pre-fetched batches are used up."""
    it = iter(loader)
    for _ in range(NUM_WORKERS * 4 + 2):
        next(it)
    t = time.time()
    for _ in range(n_batches):
        next(it)
    rate = n_batches * loader.batch_size / (time.time() - t)
    del it
    return rate


t = time.time()
for i in range(200):
    train_ds[i]
ms_per_crop = (time.time() - t) / 200 * 1000
load_speed = loader_rate(train_loader)
GPU_SPEED_M1 = 253
speed = min(load_speed, GPU_SPEED_M1)
pd.Series({
    "CPU time per augmented crop, one process (ms)": round(ms_per_crop, 1),
    "data loading speed (img/s)": round(load_speed),
    "GPU training speed in M1 (img/s)": GPU_SPEED_M1,
    "bottleneck": "GPU" if load_speed >= GPU_SPEED_M1 else "data loading",
    "estimated minutes per epoch (training part)": round(len(train_loader) * BATCH_SIZE / speed / 60, 1),
    f"estimated hours per ablation ({EPOCHS} epochs)": round(EPOCHS * len(train_loader) * BATCH_SIZE / speed / 3600, 1),
}, name="value").to_frame()


,value
"CPU time per augmented crop, one process (ms)",36.0
data loading speed (img/s),217
GPU training speed in M1 (img/s),253
bottleneck,data loading
estimated minutes per epoch (training part),8.8
estimated hours per ablation (10 epochs),1.5


## 3. Model
### 3.1 Frozen encoder + bottleneck


In [18]:
# 3.1 Frozen CLIP + bottleneck
class BottleneckHead(nn.Module):
    def __init__(self, width, dim, priv_rank):
        super().__init__()
        self.fc1 = nn.Linear(width, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.cls = nn.Linear(dim, 2)
        self.U = nn.ParameterDict({g: nn.Parameter(torch.zeros(dim, priv_rank)) for g in TRAIN_GENERATORS})
        for p in self.U.parameters():
            nn.init.normal_(p, std=0.01)

    def forward(self, z0, g=None):
        h = F.relu(self.fc1(z0))
        h = h + self.fc2(h)
        if g is not None:
            u = self.U[g]                      # (d, k)
            h = h + (h @ u) @ u.T              # private low-rank residual
        return self.cls(h), F.normalize(h, dim=-1)


torch.manual_seed(SEED)
encoder = open_clip.create_model(ARCH, pretrained=PRETRAINED, force_quick_gelu=True).visual
encoder.proj = None
encoder = encoder.to(DEVICE)
encoder.requires_grad_(False)
encoder.eval()
WIDTH = encoder.transformer.width
assert WIDTH == 1024

head = BottleneckHead(WIDTH, BOT_DIM, PRIV_RANK).to(DEVICE)
deployed = sum(p.numel() for n, p in head.named_parameters() if not n.startswith("U."))
train_n = sum(p.numel() for p in head.parameters())
display(pd.DataFrame([
    {"piece": "shared bottleneck+cls", "n": deployed, "dep": True},
    {"piece": "private U_g", "n": train_n - deployed, "dep": False},
    {"piece": "CLIP encoder", "n": 0, "dep": False},
]))
assert deployed < 104_450
print("deployed", deployed, "train", train_n)
P = deployed



,piece,n,dep
0,shared bottleneck+cls,69890,True
1,private U_g,4096,False
2,CLIP encoder,0,False


deployed 69890 train 73986


## 4. Helpers


In [19]:
# 4.1 Helpers
MEAN_GPU = torch.tensor(CLIP_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_GPU = torch.tensor(CLIP_STD, device=DEVICE).view(1, 3, 1, 1)

@torch.no_grad()
def encode_frozen(x):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        feats = encoder(x)
    return F.normalize(feats.float(), dim=-1)

def compute_loss(logits, f, y, U=None):
    ce_each = F.cross_entropy(logits, y, reduction="none")
    ce = ce_each.mean()
    n = len(y)
    i, j = torch.triu_indices(n, n, offset=1, device=f.device)
    d2 = (2 - 2 * (f @ f.T)[i, j]).clamp(min=0)
    same = y[i] == y[j]
    align = d2[same].mean() if same.any() else d2.new_zeros(())
    uniform = torch.exp(-2 * d2).mean().clamp(min=1e-6).log()
    loss = ce + W_ALIGN * align + W_UNIFORM * uniform
    if U is not None:
        loss = loss + LAMBDA * U.pow(2).sum()
    parts = {"loss": float(loss.detach()), "ce": float(ce.detach())}
    return loss, parts, ce_each.detach()

def lr_at(step, steps_per_epoch):
    cycle = CYCLE_EPOCHS * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch
    pos = step % cycle
    if pos < warm:
        return LR_MIN + (LR_MAX - LR_MIN) * pos / warm
    frac = (pos - warm) / max(cycle - warm, 1)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * frac))

def train_step(x, y, g, opt, lr):
    opt.zero_grad(set_to_none=True)
    z0 = encode_frozen(x)
    logits, h = head(z0, g=g)
    loss, parts, ce_each = compute_loss(logits, h, y, U=head.U[g])
    loss.backward()
    for pg in opt.param_groups:
        pg["lr"] = lr
    opt.step()
    return parts, ce_each

def to_model_input(uint8_batch):
    x = torch.as_tensor(uint8_batch).to(DEVICE, non_blocking=True)
    return (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU

def load_paths(paths, desc):
    out = np.empty((len(paths), 224, 224, 3), dtype=np.uint8)
    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))
    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out

def ram_images(name, df):
    if images.get(name) is None:
        images[name] = load_images(df, f"{name} crops -> RAM")
    return images[name]

@torch.no_grad()
def predict_probs(arr, desc="predict"):
    head.eval(); out = []
    for i in tqdm(range(0, len(arr), EVAL_BATCH), desc=desc, leave=False):
        z0 = encode_frozen(to_model_input(arr[i:i + EVAL_BATCH]))
        logits, _ = head(z0, g=None)  # discard private
        out.append(logits.float().softmax(1)[:, 1].cpu())
    head.train()
    return torch.cat(out).numpy()

def auc(labels, scores):
    labels = np.asarray(labels)
    ranks = pd.Series(np.asarray(scores)).rank().values
    n_pos, n_neg = labels.sum(), len(labels) - labels.sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def metrics_table(pred):
    groups = {"seen combined (" + "+".join(SHORT[g] for g in TRAIN_GENERATORS) + ")": TRAIN_GENERATORS}
    groups.update({g: [g] for g in ALL_GENERATORS})
    rows = []
    for name, gens in groups.items():
        sub = pred[pred.generator.isin(["original", *gens])]
        if (sub.label == 1).sum() == 0: continue
        vid = sub.groupby(["generator", "video_id"], as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
        rows.append({"test set": name, "status": "seen" if set(gens) <= set(TRAIN_GENERATORS) else "UNSEEN",
                     "frame AUC": 100 * auc(sub.label, sub.prob), "video AUC": 100 * auc(vid.label, vid.prob),
                     "video acc @0.5": 100 * float(((vid.prob > 0.5) == vid.label).mean())})
    return pd.DataFrame(rows).round(2)

def video_auc(index_df, probs):
    vid = index_df.assign(prob=probs).groupby("video_id", as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
    return 100 * auc(vid.label, vid.prob)

def frame_pred(df, probs):
    out = df[["split", "label", "generator", "video_id", "frame_idx", "path"]].copy(); out["prob"] = probs; return out

def save_atomic(obj, path):
    tmp = path.with_name(path.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)

def load(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def run_dir():
    return M15B_DIR / f"m15b_arch4b_d{BOT_DIM}_k{PRIV_RANK}_{ARCH}_seed{SEED}"

def check_config(run_dir, config):
    path = run_dir / "config.json"
    dumped = json.loads(json.dumps(config))
    if path.exists():
        assert json.loads(path.read_text()) == dumped, f"{run_dir} differs; delete to restart"
    else:
        path.write_text(json.dumps(dumped, indent=1))

def head_state():
    return {k: v.detach().cpu() for k, v in head.state_dict().items()}



### 4.2 Smoke


In [20]:
# 4.2 Smoke
check_rows = pd.concat([train_df[train_df.generator == "original"].iloc[:4],
                        train_df[train_df.generator == TRAIN_GENERATORS[0]].iloc[:4]])
check_ds = FaceDataset(check_rows.reset_index(drop=True), train=False, images=None)
xs, ys = zip(*[(check_ds[i][0], check_ds[i][1]) for i in range(len(check_ds))])
check_x, check_y = torch.stack(xs).to(DEVICE), torch.stack(ys).to(DEVICE)
g0 = TRAIN_GENERATORS[0]
opt = torch.optim.Adam(head.parameters(), lr=LR_MAX)
before = head.U[g0].detach().clone()
train_step(check_x, check_y, g0, opt, LR_MAX)
assert not torch.equal(before, head.U[g0])
# encoder frozen
for p in encoder.parameters():
    assert p.grad is None or float(p.grad.abs().sum()) == 0
print("M15b SMOKE OK | U_g moved | encoder frozen")
torch.cuda.empty_cache()



M15b SMOKE OK | U_g moved | encoder frozen


## 5. Train


In [21]:
# 5.1 Train
def train_arch4b():
    rd = run_dir(); rd.mkdir(parents=True, exist_ok=True)
    dep = sum(p.numel() for n, p in head.named_parameters() if not n.startswith("U."))
    check_config(rd, {**COMMON_CONFIG, "deployed_trainable": dep, "train_trainable": sum(p.numel() for p in head.parameters())})
    opt = torch.optim.Adam(head.parameters(), lr=LR_MAX)
    history, start_epoch = [], 0
    if (rd / "last.pt").exists():
        ckpt = load(rd / "last.pt")
        head.load_state_dict({k: v.to(DEVICE) for k, v in ckpt["head"].items()})
        opt.load_state_dict(ckpt["adam"])
        history, start_epoch = ckpt["history"], ckpt["epoch"]
        print("resumed", start_epoch)
    print(f"===== Arch4B d={BOT_DIM} k={PRIV_RANK} dep={dep:,} {start_epoch}/{EPOCHS} =====")
    steps_per_epoch = len(train_loader)
    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch); head.train()
        sums, n_steps, t0 = defaultdict(float), 0, time.time()
        ce_sum = torch.zeros(len(CATEGORIES), device=DEVICE)
        ce_cnt = torch.zeros(len(CATEGORIES), device=DEVICE)
        for i, (x, y, gen) in enumerate(tqdm(train_loader, desc=f"4B ep {epoch+1}/{EPOCHS}", leave=False)):
            x, y, gen = (t.to(DEVICE, non_blocking=True) for t in (x, y, gen))
            fake_codes = gen[gen != GEN_CODE["original"]]
            assert len(fake_codes) > 0 and torch.all(fake_codes == fake_codes[0])
            g = CATEGORIES[int(fake_codes[0])]
            lr = lr_at(epoch * steps_per_epoch + i, steps_per_epoch)
            parts, ce_each = train_step(x, y, g, opt, lr)
            for k, v in parts.items(): sums[k] += v
            ce_sum += torch.bincount(gen, weights=ce_each, minlength=len(CATEGORIES))
            ce_cnt += torch.bincount(gen, minlength=len(CATEGORIES)); n_steps += 1
        val_table = metrics_table(frame_pred(val_df, predict_probs(ram_images("val", val_df), "val")))
        row = {"epoch": epoch + 1, "lr (end)": float(lr)}
        row.update({k: v / n_steps for k, v in sums.items()})
        row.update({f"|U| {SHORT[g]}": float(head.U[g].norm()) for g in TRAIN_GENERATORS})
        row.update({"val video AUC": float(val_table["video AUC"].iloc[0]), "minutes": (time.time() - t0) / 60})
        history.append(row)
        save_atomic(head_state(), rd / f"epoch_{epoch+1:02d}.pt")
        save_atomic({"epoch": epoch + 1, "head": head_state(), "adam": opt.state_dict(), "history": history}, rd / "last.pt")
        print(" | ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in row.items()))
    if not (rd / "results.json").exists():
        table = metrics_table(frame_pred(test_df, predict_probs(ram_images("test", test_df), "test")))
        display(table)
        res = {"architecture": "Arch4B", "notebook": "M15b", "deployed_trainable": dep,
               "test": table.to_dict(orient="records"), "history": history,
               "U_norms": {SHORT[g]: float(head.U[g].norm()) for g in TRAIN_GENERATORS}}
        for name, flag, index_path, root in [("dfd", RUN_DFD, DFD_INDEX, FACES_DFD), ("uadfv", RUN_UADFV, UADFV_INDEX, FACES_UADFV)]:
            if not flag: continue
            idx = pd.read_csv(index_path, dtype={"video_id": str})
            arr = load_paths([root / p for p in idx.path], f"{name}")
            res[f"{name}_video_auc"] = video_auc(idx, predict_probs(arr, name)); del arr
            print(name.upper(), res[f"{name}_video_auc"])
        (rd / "results.json").write_text(json.dumps(res, indent=1))
    torch.cuda.empty_cache()

train_arch4b(); print("M15b finished")



resumed 10
===== Arch4B d=64 k=16 dep=69,890 10/10 =====
M15b finished


## 6. Decision


In [22]:
# 6.1 Decision
def video_auc_from_records(records):
    t = pd.DataFrame(records).set_index("test set")
    row = {"seen": float(t.iloc[0]["video AUC"])}
    row.update({SHORT[g]: float(t.loc[g, "video AUC"]) for g in ALL_GENERATORS})
    return row
rows = [{"model": "A0 SB-LN+linear", "trainable": 36418, **A0, "UADFV": float("nan")}]
for name, (folder, dfd, uadfv) in REFS.items():
    if not (folder / "results.json").exists():
        print("missing", name); continue
    res = json.loads((folder / "results.json").read_text())
    rows.append({"model": name, "trainable": res.get("deployed_trainable", P),
                 **video_auc_from_records(res["test"]),
                 "DFD": res.get("dfd_video_auc", dfd), "UADFV": res.get("uadfv_video_auc", uadfv)})
res = json.loads((run_dir() / "results.json").read_text())
arch = {"model": "Arch4B (M15b)", "trainable": res["deployed_trainable"],
        **video_auc_from_records(res["test"]),
        "DFD": res.get("dfd_video_auc", float("nan")), "UADFV": res.get("uadfv_video_auc", float("nan"))}
rows.append(arch)
main_table = pd.DataFrame(rows).set_index("model"); display(main_table.round(2))
main_table.to_csv(M15B_DIR / "main_table.csv")
keep = float(arch["NT"]) >= A0["NT"] and float(arch["DFD"]) >= 95.5
print(f"NT={arch['NT']:.2f} DFD={arch['DFD']:.2f}")
print("DECISION:", "KEEP Arch4B" if keep else "REJECT Arch4B")
(M15B_DIR / "decision.json").write_text(json.dumps({
    "architecture": "Arch4B", "keep": keep,
    "arch": {k: arch[k] for k in ("seen", "NT", "FSh", "DFD", "trainable")},
    "delta_vs_A0": {k: round(float(arch[k]) - A0[k], 2) for k in ("seen", "NT", "FSh", "DFD")},
}, indent=2))



,trainable,seen,DF,F2F,FS,NT,FSh,DFD,UADFV
model,,,,,,,,,
A0 SB-LN+linear,36418,94.54,99.49,97.86,99.60,81.21,92.25,95.15,NaN
full LayerNorms (M4 λ=∞),69890,96.41,99.72,98.66,99.34,87.90,90.53,96.05,99.38
GBLA λ=0.01 (M4) / Arch1,69890,95.55,99.43,98.55,99.17,85.06,92.91,96.71,99.63
Arch4B (M15b),69890,88.85,98.05,90.61,97.21,69.54,81.98,88.59,97.13


NT=69.54 DFD=88.59
DECISION: REJECT Arch4B


268